# Notebook 19 — compact profiler visual review

This supplemental notebook reads the already committed whole-JEPA profile and makes its long CUDA entry names readable. It preserves every frozen scientific export, the active main consumer and core run artifacts. It neither starts training nor accesses a GPU or any image.

Inputs: `artifacts/notebook_19/<RUN_ID>/profiling/profile_result.json`. Outputs: a compact native PNG, exact full-name/value CSV and source SHA receipt under `artifacts/notebook_19/<RUN_ID>/visual_review/` only. Expected runtime: under 30 seconds in a fresh `ahr-go` kernel. Checkpoints: no model state is read or written. Exact resume: set the same `ROOT` and `RUN_ID` below and click Run All; presentation output is atomically refreshed. A missing or incomplete committed profile raises an explicit error.

Each row retains an entry ID mapped to the CSV. GPU qualification applies only to the recorded measurement window. Profiler entries can include both framework and CUDA kernel rows; their accumulated self-device times must not be summed into wall time. Real data decoding and checkpoint costs are excluded from this profile. This figure adds no accuracy or superiority claim.

### Reusable presentation function

Other review notebooks may load only this explicitly tagged definition cell. No upstream training cell is imported.

In [ ]:
from pathlib import Path
import csv,hashlib,io,json,math,os,re,textwrap,time

def review_profile(ROOT,RUN_ID):
    """Render committed profile entries only; write exclusively to this run's visual_review/."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    ROOT=Path(ROOT).resolve();RUN_ID=str(RUN_ID)
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*',RUN_ID):raise ValueError('RUN_ID must be a single safe run-directory name')
    source=ROOT/'artifacts/notebook_19'/RUN_ID/'profiling/profile_result.json'
    payload=source.read_bytes();profile=json.loads(payload);source_sha=hashlib.sha256(payload).hexdigest()
    if profile.get('status')!='COMPLETE':raise ValueError('Read only a committed COMPLETE profiling result; preserve uncompleted results as UNKNOWN')
    destination=source.parent.parent/'visual_review';destination.mkdir(parents=True,exist_ok=True)
    measured=[row for row in profile.get('rows',[])if row.get('status')=='MEASURED']
    if not measured:raise ValueError('The committed profile has no measured operator entries')
    def finite(value):return isinstance(value,(int,float))and not isinstance(value,bool)and math.isfinite(value)
    def atomic_text(path,text):
        temporary=path.with_name(path.name+'.tmp.'+str(os.getpid()))
        with temporary.open('w')as handle:handle.write(text);handle.flush();os.fsync(handle.fileno())
        os.replace(temporary,path)
    def compact(name):
        # Stable entry IDs, rather than shortened names, retain the exact lookup identity.
        if 'AttentionBackwardKernel'in name:return 'CUDA attention backward'
        if 'AttentionKernel'in name:return 'CUDA attention forward'
        if 'indexing_backward_kernel'in name:return 'CUDA indexing backward'
        if name.startswith('aten::'):return textwrap.shorten(name,width=42,placeholder='…')
        return textwrap.shorten(name.split('(')[0].split('<')[0],width=42,placeholder='…')
    labels={'vit':'ViT','original18':'Original learned implementation','optimized19':'Optimized learned implementation, local 50%','optimized19_gate01':'Optimized learned implementation, local 10%'}
    colors={'vit':'#315f8c','original18':'#697880','optimized19':'#c27228','optimized19_gate01':'#6a6f32'}
    entries=[];groups=[]
    for row in measured:
        variant=row.get('variant','UNKNOWN');operations=[]
        for op in row.get('operators',[]):
            value=op.get('self_device_ms')
            if finite(value):operations.append((str(op.get('operator','UNKNOWN')),float(value),op))
        operations.sort(key=lambda value:(-value[1],value[0]))
        qualified=(row.get('performance_claim_qualified')is True and row.get('performance_status')=='MEASURED_EXCLUSIVE_OBSERVED')
        group=[]
        for index,(name,value,op)in enumerate(operations,1):
            record={'variant':variant,'entry_id':variant+'-P'+str(index).zfill(2),'compact_operator_label':compact(name),'full_operator_name':name,
              'self_device_ms':value,'calls':op.get('calls'),'self_cpu_ms':op.get('self_cpu_ms'),'device_total_ms':op.get('device_total_ms'),
              'profile_status':row.get('status'),'performance_status':row.get('performance_status'),'performance_claim_qualified':qualified,
              'source_profile_sha256':source_sha,'measurement_scope':row.get('measurement_scope','UNKNOWN'),'profile_role':row.get('profile_role','UNKNOWN')}
            entries.append(record);group.append(record)
        if group:groups.append((row,group[:6]))
    if not groups:raise ValueError('No finite committed profiler self-device measurements are available')
    plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.spines.top':False,'axes.spines.right':False})
    figure,axes=plt.subplots(len(groups),1,figsize=(11.4,3.25*len(groups)+.55),squeeze=False,layout='constrained')
    common_maximum=max(item['self_device_ms']for row,group in groups for item in group)
    for ax,(row,group)in zip(axes[:,0],groups):
        variant=row.get('variant','UNKNOWN');color=colors.get(variant,'#697880');qualified=all(item['performance_claim_qualified']for item in group)
        values=[item['self_device_ms']for item in group];ticks=[item['entry_id'].split('-')[-1]+' '+item['compact_operator_label']for item in group]
        bars=ax.barh(range(len(group)),values,color=color,height=.64,hatch=None if qualified else'//')
        ax.invert_yaxis();ax.set_yticks(range(len(group)),ticks);ax.set_xlim(0,common_maximum*1.18 if common_maximum>0 else 1.)
        for bar,value in zip(bars,values):ax.text(value+max(values)*.018,bar.get_y()+bar.get_height()/2,f'{value:.1f}',va='center',fontsize=10)
        ax.set_xlabel('Accumulated self device time (ms)');ax.grid(axis='x',alpha=.18)
        title=labels.get(variant,variant)+'\n'+('GPU observation qualified for this recorded window'if qualified else'UNQUALIFIED performance window — descriptive entries only')
        ax.set_title(title,fontsize=12,pad=10)
    figure.suptitle('Complete JEPA profiler: six largest entries per implementation\nSynthetic fixed inputs; accumulated profiler time is not additive wall time',fontsize=13)
    image=destination/'operator_costs_compact.png';temporary=image.with_name('operator_costs_compact.tmp.'+str(os.getpid())+'.png')
    figure.savefig(temporary,dpi=145,bbox_inches='tight');plt.close(figure);os.replace(temporary,image)
    columns=list(entries[0]);stream=io.StringIO();writer=csv.DictWriter(stream,fieldnames=columns);writer.writeheader();writer.writerows(entries)
    table=destination/'operator_full_names.csv';atomic_text(table,stream.getvalue())
    receipt={'schema':'fmm19-supplemental-profile-review/v1','status':'COMPLETE','run_id':RUN_ID,'generated_unix':time.time(),
      'input':{'path':str(source),'sha256':source_sha},'measured_implementations':len(groups),'all_profile_entries_exported':len(entries),
      'qualified_implementations':sum(all(item['performance_claim_qualified']for item in group)for row,group in groups),
      'figure':{'path':str(image),'sha256':hashlib.sha256(image.read_bytes()).hexdigest()},'full_names_csv':{'path':str(table),'sha256':hashlib.sha256(table.read_bytes()).hexdigest()},
      'official_test_pixels_read':0,'training_launched':False,'scientific_sources_edited':False,'scope':'Presentation-only review of committed measurements; no speed, accuracy or superiority inference is added'}
    if source.read_bytes()!=payload:raise RuntimeError('Committed profiling input changed during presentation; do not claim a stable review')
    atomic_text(destination/'review_receipt.json',json.dumps(receipt,indent=2,allow_nan=False))
    return receipt


### Review the committed run

The full names remain available in the CSV so shortened labels stay auditable.

In [ ]:
from IPython.display import display,Image,Markdown
ROOT=Path('/home/benjamin/workspace/ahr_go/fmm_jeppa')
RUN_ID='compute-gate-002'
REVIEW=review_profile(ROOT,RUN_ID)
display(Image(filename=REVIEW['figure']['path']))
display(Markdown('Full operator identities and exact values: [operator_full_names.csv]('+REVIEW['full_names_csv']['path']+'). Entry IDs match each plot row.'))
print(json.dumps({'status':REVIEW['status'],'implementations':REVIEW['measured_implementations'],'qualified_windows':REVIEW['qualified_implementations'],'full_operator_rows':REVIEW['all_profile_entries_exported'],'official_test_pixels_read':0},indent=2))


### Verify exact value bindings and unchanged inputs

In [ ]:
# The displayed source values must exactly match the full-name CSV; inputs stay byte-identical.
with Path(REVIEW['full_names_csv']['path']).open(newline='')as handle:csv_rows=list(csv.DictReader(handle))
profile=json.loads(Path(REVIEW['input']['path']).read_text())
source_entries={(row['variant'],op['operator']):op['self_device_ms']for row in profile['rows']for op in row.get('operators',[])if isinstance(op.get('self_device_ms'),(int,float))and math.isfinite(op['self_device_ms'])}
assert len(csv_rows)==REVIEW['all_profile_entries_exported']
for row in csv_rows:assert abs(float(row['self_device_ms'])-source_entries[(row['variant'],row['full_operator_name'])])<1e-10
assert hashlib.sha256(Path(REVIEW['input']['path']).read_bytes()).hexdigest()==REVIEW['input']['sha256']
assert Path(REVIEW['figure']['path']).parent.name=='visual_review'
print('PASS: CSV values and complete names reconcile with all committed profiler entries; profiling input is unchanged.')
